In [2]:
import os, shutil, zipfile
from pathlib import Path

In [7]:
def unzip_data(zip_path: str, extract_to: str):
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"[INFO] Unzipped {zip_path} to {extract_to}")
    except Exception as e:
        print(f"[ERROR] Failed to unzip {zip_path}: {e}")

ZIP_PATH = "/content/ocr_processed_complete_final.zip"
EXTRACT_TO = "/content/data"

unzip_data(ZIP_PATH, EXTRACT_TO)

[INFO] Unzipped /content/ocr_processed_complete_final.zip to /content/data


In [3]:
colab_content = "/content/"
folders = [
    "models",
    "configs",
    "en_PP-OCRv3_rec_slim_train"
]

# Create them
for folder in folders:
    full_path = os.path.join(colab_content, folder)
    os.makedirs(full_path, exist_ok=True)
    print(f"[INFO] Created: {full_path}")

[INFO] Created: /content/models
[INFO] Created: /content/configs
[INFO] Created: /content/en_PP-OCRv3_rec_slim_train


In [4]:
shutil.rmtree('/content/sample_data')

In [15]:
label_dir = ('/content/data/ocr_processed/labels')
label_files = ['train_final.txt', 'valid_final.txt', 'test_final.txt']

for label_file in label_files:
    label_path = os.path.join(label_dir, label_file)

    with open(label_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    updated_lines = []
    for line in lines:
        if not line.strip():
            continue
        img_path, label = line.strip().split('\t')
        corrected_img_path = img_path.replace('.png', '.jpg')
        updated_lines.append(f'{corrected_img_path}\t{label}')

    with open(label_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(updated_lines) + '\n')

print("✅ All .jpeg extensions in label files have been corrected to .jpg.")

✅ All .jpeg extensions in label files have been corrected to .jpg.


In [19]:
def remove_invalid_label_entries(label_path: str, image_root: str) -> int:
    """
    Removes entries in the label file whose corresponding image is missing.

    Args:
        label_path (str): Path to the label .txt file.
        image_root (str): Root folder path where the images are stored.

    Returns:
        int: Number of entries removed.
    """
    with open(label_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    cleaned_lines = []
    removed_count = 0

    for line in lines:
        if not line.strip():
            continue
        img_rel_path = line.strip().split('\t')[0]
        img_abs_path = os.path.join(image_root, os.path.basename(img_rel_path))

        if os.path.isfile(img_abs_path):
            cleaned_lines.append(line)
        else:
            removed_count += 1

    # Overwrite with cleaned lines
    with open(label_path, 'w', encoding='utf-8') as f:
        f.writelines(cleaned_lines)

    return removed_count


def check_label_file_integrity(label_path: str, image_root: str) -> bool:
    """
    Checks if all label entries point to existing images.

    Args:
        label_path (str): Path to the label .txt file.
        image_root (str): Root folder path where the images are stored.

    Returns:
        bool: True if all entries are valid, False otherwise.
    """
    with open(label_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    missing_files = []
    for line in lines:
        if not line.strip():
            continue
        img_rel_path = line.strip().split('\t')[0]
        img_abs_path = os.path.join(image_root, os.path.basename(img_rel_path))

        if not os.path.isfile(img_abs_path):
            missing_files.append(img_abs_path)

    if missing_files:
        print(f"\n❌ {len(missing_files)} missing images for {os.path.basename(label_path)}:")
        for path in missing_files[:10]:
            print(f"   - {path}")
        if len(missing_files) > 10:
            print("   ... and more.")
        return False
    else:
        print(f"\n✅ All entries in {os.path.basename(label_path)} are valid.")
        return True


base_path = '/content/data/ocr_processed'
label_dir = os.path.join(base_path, 'labels')
dataset_dirs = {
    'train_final.txt': os.path.join(base_path, 'cropped', 'train'),
    'valid_final.txt': os.path.join(base_path, 'cropped', 'valid'),
    'test_final.txt': os.path.join(base_path, 'cropped', 'test'),
}

for label_file, image_dir in dataset_dirs.items():
    label_path = os.path.join(label_dir, label_file)

    print(f"\n🔎 Checking integrity of {label_file}...")
    ok = check_label_file_integrity(label_path, image_dir)

    if not ok:
        removed = remove_invalid_label_entries(label_path, image_dir)
        print(f"🧹 Removed {removed} invalid entries from {label_file}.")


🔎 Checking integrity of train_final.txt...

❌ 8 missing images for train_final.txt:
   - /content/data/ocr_processed/cropped/train/6bf2730d-82bf-43ac-8802-b61315d85b96___2017-Skoda-Octavia-rear-high-revealed-for-India-cropped.jpg_0.jpg
   - /content/data/ocr_processed/cropped/train/00b42b2c-f193-4863-b92c-0245cbc816da___3e7fd381-0ae5-4421-8a70-279ee0ec1c61_Nissan-Terrano-Petrol-Review-cropped-Black-Front-Angle_0.jpg
   - /content/data/ocr_processed/cropped/train/018b52e6-e9a1-42c2-8ce7-0617e8c8e021___3e7fd381-0ae5-4421-8a70-279ee0ec1c61_sbtb02_auto1_0.JPG
   - /content/data/ocr_processed/cropped/train/HP18_0.jpg
   - /content/data/ocr_processed/cropped/train/a970d6e0-5948-4412-a1ec-67c5240722a8___cropped-gallery-misc4-250x250.jpg_0.jpg
   - /content/data/ocr_processed/cropped/train/JH1_0.jpg
   - /content/data/ocr_processed/cropped/train/6d69b523-72b7-4be9-a79f-428ef8148dcd___81auoS-H8eL._SX425_.jpg_0.jpg
   - /content/data/ocr_processed/cropped/train/e750b959-1b5a-41bf-bf64-d8099089e

In [17]:
!pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

!git clone https://github.com/PaddlePaddle/PaddleOCR
%cd PaddleOCR
!pip install -r requirements.txt

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 268691, done.
remote: Counting objects: 100% (583/583), done.
remote: Compressing objects: 100% (133/133), done.
^C
[Errno 2] No such file or directory: 'PaddleOCR'
/content/PaddleOCR


In [6]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [8]:
%pwd

'/content/PaddleOCR'

Run Training

In [20]:
!python tools/train.py -c /content/configs/en_PP-OCRv3_mobile_rec.yml

/usr/local/lib/python3.11/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2025/08/06 20:14:38] ppocr WARNING: Skipping import of the encryption module.
[2025/08/06 20:14:38] ppocr INFO: Architecture : 
[2025/08/06 20:14:38] ppocr INFO:     Backbone : 
[2025/08/06 20:14:38] ppocr INFO:         last_conv_stride : [1, 2]
[2025/08/06 20:14:38] ppocr INFO:         last_pool_kernel_size : [2, 2]
[2025/08/06 20:14:38] ppocr INFO:         last_pool_type : avg
[2025/08/06 20:14:38] ppocr INFO:         name : MobileNetV1Enhance
[2025/08/06 20:14:38] ppocr INFO:         pretrained : /content/configs/en_PP-OCRv3_rec_slim_train/best_accuracy
[2025/08/06 20:14:38] ppocr INFO:         scale : 0.5
[2025/08/06 20:14:38] ppocr INFO:     Head : 
[2025/08/06 2

In [10]:
shutil.rmtree('/content/runs')

Evaluate

In [ ]:
!python tools/eval.py -c /content/configs/en_PP-OCRv3_mobile_rec.yml \
-o Global.checkpoints=/content/output/v3_en_mobile/latest

Export Inference Model

In [ ]:
!python tools/export_model.py -c /content/configs/en_PP-OCRv3_mobile_rec.yml \
-o Global.pretrained_model=/content/output/rec_lp/best_accuracy \
Global.save_inference_dir=/content/output/inference/rec_lp/

In [ ]:
# Download logs
!zip -r rec_lp_logs.zip /content/output/rec_lp/

# # Or run visualdl locally on your machine
# visualdl --logdir ./output/rec_lp/

In [ ]:
from google.colab import files

def runs_collab(folder_name="/content/ocr_processed", output_name="ocr_processed_complete_final.zip"):
    if not os.path.exists(folder_name):
        print(f"[ERROR] Folder '{folder_name}' does not exist.")
        return

    # Create the zip file
    print(f"[INFO] Zipping folder '{folder_name}'...")
    os.system(f"zip -r {output_name} {folder_name}")

    # Download the zip file
    if os.path.exists(output_name):
        print(f"[INFO] Downloading '{output_name}'...")
        files.download(output_name)
    else:
        print("[ERROR] Failed to create zip file.")

runs_collab()